# Validierung der erzeugten STIX-2.1-Bundles 

Die erzeugten Bundles werden mit dem STIX-Validator der OASIS-Community
gegen die STIX-2.1-Spezifikation geprüft.

Für einen Testlauf kann `LIMIT` zunächst auf einen kleinen Wert gesetzt
werden. Für die vollständige Validierung wird `LIMIT = None` verwendet.

In [1]:
%pip install -q stix2-validator


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Schemas sicherstellen
Die pip-Version bringt die JSON-Schemas teilweise nicht mit. Diese Zelle prüft das und holt sie bei Bedarf einmalig von GitHub.

In [2]:
import os, shutil, subprocess, tempfile
import stix2validator

pkg_dir = os.path.dirname(stix2validator.__file__)
schema_target = os.path.join(pkg_dir, "schemas-2.1", "schemas")

if os.path.isdir(schema_target):
    print("Schemas vorhanden:", schema_target)
else:
    print("Schemas fehlen, lade sie von GitHub ...")
    tmp = tempfile.mkdtemp()
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/oasis-open/cti-stix2-json-schemas.git", tmp],
                   check=True, capture_output=True)
    os.makedirs(os.path.dirname(schema_target), exist_ok=True)
    shutil.copytree(os.path.join(tmp, "schemas"), schema_target)
    shutil.rmtree(tmp, ignore_errors=True)
    print("Schemas installiert nach:", schema_target)

Schemas vorhanden: /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/stix2validator/schemas-2.1/schemas


## Einstellungen

In [ ]:
from pathlib import Path

BUNDLE_DIR = Path("data/bundles_final")   # Bundle-Verzeichnis
LIMIT = 50                           # None = alle Bundles
STIX_VERSION = "2.1"

def variante(dateiname: str):
    #ermittelt die Variante anhand des Dateinamens
    return "angereichert" if "enriched" in dateiname.lower() else "basis"

files = sorted(BUNDLE_DIR.rglob("*.json"))
if LIMIT:
    files = files[:LIMIT]
print(f"{len(files)} Bundles gefunden in {BUNDLE_DIR}")
if not files:
    raise FileNotFoundError(f"Keine .json-Dateien in {BUNDLE_DIR.resolve()} gefunden. BUNDLE_DIR prüfen.")

50 Bundles gefunden in data/bundles_final


## Validierung

In [5]:
import re, time
from collections import Counter
from stix2validator import ValidationOptions, validate_file

CODE_RE = re.compile(r"\{(\d+)\}")
code_of = lambda msg: (CODE_RE.search(msg).group(1) if CODE_RE.search(msg) else "?")

options = ValidationOptions(version=STIX_VERSION)   # Standardmodus

rows = []
error_codes, warning_codes = Counter(), Counter()
warning_examples, error_examples = {}, {}
start = time.time()

for i, path in enumerate(files, 1):
    res = validate_file(str(path), options)
    errors, warnings = [], []
    for obj_res in (res.object_results or []):
        errors += [e.message for e in (obj_res.errors or [])]
        warnings += [str(w) for w in (obj_res.warnings or [])]
    if res.fatal:
        errors.append(f"{{fatal}} {res.fatal}")

    for e in errors:
        error_codes[code_of(e)] += 1
        error_examples.setdefault(code_of(e), e[:200])
    for w in warnings:
        warning_codes[code_of(w)] += 1
        warning_examples.setdefault(code_of(w), w.split(": ", 1)[-1][:160])

    rows.append({"datei": path.name, "variante": variante(path.name),
                 "gueltig": bool(res.is_valid), "fehler": len(errors),
                 "warnungen": len(warnings)})
    if i % 25 == 0 or i == len(files):
        print(f"  {i}/{len(files)} geprüft ({time.time() - start:.0f} s)", flush=True)

laufzeit = time.time() - start
print(f"Fertig in {laufzeit:.0f} s")

  25/50 geprüft (19 s)
  50/50 geprüft (37 s)
Fertig in 37 s


## Zusammenfassung

In [6]:
n = len(rows)
ok = sum(r["gueltig"] for r in rows)
print(f"Geprüfte Bundles: {n}")
print(f"Spezifikationskonform (keine Fehler): {ok} von {n}")
for v in ("basis", "angereichert"):
    sub = [r for r in rows if r["variante"] == v]
    if sub:
        print(f"  {v}: {sum(r['gueltig'] for r in sub)} von {len(sub)}")

invalid = [r["datei"] for r in rows if not r["gueltig"]]
if invalid:
    print(f"\nBundles mit Fehlern ({len(invalid)}), erste 10: {invalid[:10]}")
    print("\nFehler nach Prüfcode (Anzahl, Beispiel):")
    for c, cnt in error_codes.most_common():
        print(f"  {{{c}}}: {cnt:>7}  {error_examples[c]}")

print("\nWarnungen nach Prüfcode (Anzahl, Beispiel):")
for c, cnt in warning_codes.most_common():
    print(f"  {{{c}}}: {cnt:>7}  {warning_examples[c]}")

print(f"\nValidator-Version: {getattr(stix2validator, '__version__', 'siehe pip show stix2-validator')}")
print(f"Laufzeit: {laufzeit:.0f} s")

Geprüfte Bundles: 50
Spezifikationskonform (keine Fehler): 50 von 50
  basis: 25 von 25
  angereichert: 25 von 25

Warnungen nach Prüfcode (Anzahl, Beispiel):
  {103}:   26737  {103} Given ID value identity--6a4bb6a6-f976-5a01-b3a9-e28cd5d77769 is not a valid UUIDv4 ID.
  {202}:   25355  {202} 'software' is permitted, but not a suggested relationship source object for the 'has' relationship.
  {401}:     175  {401} Custom property 'x_wid_status' should be implemented using an extension with an 'extension_type' of 'property-extension' or 'toplevel-property-extension'.
  {302}:      50  {302} External reference 'WID-Portal' has a URL but no hash.

Validator-Version: 3.3.1
Laufzeit: 37 s


## Einzelergebnisse speichern

In [9]:
import csv
with open("validation_results.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys(), delimiter=";")
    w.writeheader()
    w.writerows(rows)
print("Gespeichert: validation_results_fails.csv")

Gespeichert: validation_results_fails.csv
